# Exploración de datos — Cloud Provider Analytics

**Minería de Datos II · ISTEA · Primera entrega**

En este notebook leemos por primera vez las fuentes del proyecto con PySpark para conocer cómo vienen los datos:
cuántas filas tienen, qué columnas traen nulos, si hay duplicados y qué problemas de calidad aparecen.

No limpiamos ni guardamos nada: solo **observamos**. Lo que encontremos acá se usa para diseñar las capas Bronze y Silver.

> Para ejecutarlo: **Entorno de ejecución → Ejecutar todas**.

## 1. Instalar PySpark

Igual que en el notebook de clase: Colab ya trae Java, solo falta instalar PySpark.

In [ ]:
!pip -q install pyspark

## 2. Obtener los datos

Clonamos el repositorio del proyecto y descomprimimos el dataset en `/content/datos`.
El zip original no se modifica: es nuestra zona **Landing** (datos crudos inmutables).

In [ ]:
!git clone -q https://github.com/damiann8n/cloud-provider-analytics.git
!unzip -q -o cloud-provider-analytics/data/cloud_provider_challenge_dataset_v1.zip -d /content/datos
!ls /content/datos/datalake/landing

In [ ]:
# Guardamos la ruta en una variable para no repetirla en cada lectura
LANDING = "/content/datos/datalake/landing"

## 3. Crear la sesión de Spark

- `local[*]`: usa todos los núcleos de la máquina de Colab.
- `shuffle.partitions = 8`: el dataset es chico, no necesitamos las 200 particiones por defecto.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("exploracion-cloud-provider")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

print("Versión de Spark:", spark.version)

## 4. Leer las fuentes batch (CSV)

Leemos los 7 archivos CSV. En esta etapa **no** usamos `inferSchema`: así todas las columnas llegan como texto
y podemos ver los valores tal cual vienen, sin que Spark los transforme.

> En Bronze sí vamos a definir un esquema explícito para cada fuente.

In [ ]:
orgs      = spark.read.option("header", True).csv(f"{LANDING}/customers_orgs.csv")
users     = spark.read.option("header", True).csv(f"{LANDING}/users.csv")
resources = spark.read.option("header", True).csv(f"{LANDING}/resources.csv")
tickets   = spark.read.option("header", True).csv(f"{LANDING}/support_tickets.csv")
marketing = spark.read.option("header", True).csv(f"{LANDING}/marketing_touches.csv")
nps       = spark.read.option("header", True).csv(f"{LANDING}/nps_surveys.csv")
billing   = spark.read.option("header", True).csv(f"{LANDING}/billing_monthly.csv")

### 4.1 Cantidad de filas por fuente

In [ ]:
print("customers_orgs   :", orgs.count())
print("users            :", users.count())
print("resources        :", resources.count())
print("support_tickets  :", tickets.count())
print("marketing_touches:", marketing.count())
print("nps_surveys      :", nps.count())
print("billing_monthly  :", billing.count())

### 4.2 Una mirada rápida a cada tabla

In [ ]:
orgs.show(5, truncate=False)

In [ ]:
billing.show(5, truncate=False)

In [ ]:
tickets.show(5, truncate=False)

## 5. Nulos por columna

Definimos una función chica que recorre las columnas y cuenta cuántos valores nulos tiene cada una.
La usamos con todas las tablas para no repetir el mismo código siete veces.

In [ ]:
def contar_nulos(df, nombre):
    total = df.count()
    print(f"--- {nombre} ({total} filas)")
    for columna in df.columns:
        nulos = df.filter(F.col(columna).isNull()).count()
        if nulos > 0:
            print(f"   {columna}: {nulos} nulos ({nulos / total:.1%})")

In [ ]:
contar_nulos(orgs, "customers_orgs")
contar_nulos(users, "users")
contar_nulos(resources, "resources")
contar_nulos(tickets, "support_tickets")
contar_nulos(marketing, "marketing_touches")
contar_nulos(nps, "nps_surveys")
contar_nulos(billing, "billing_monthly")

Solo se listan las columnas con nulos. Hay que distinguir dos casos:
- **Nulos esperables:** `resolved_at` en un ticket abierto o `last_login` en un usuario que nunca entró.
- **Nulos de calidad:** `nps_score`, `csat` o `credits`, donde falta un dato que debería existir.

## 6. Duplicados por clave

Si una clave (por ejemplo `org_id`) es única, la cantidad de valores distintos debe ser igual a la cantidad de filas.

In [ ]:
print("orgs      duplicados:", orgs.count()      - orgs.select("org_id").distinct().count())
print("users     duplicados:", users.count()     - users.select("user_id").distinct().count())
print("resources duplicados:", resources.count() - resources.select("resource_id").distinct().count())
print("tickets   duplicados:", tickets.count()   - tickets.select("ticket_id").distinct().count())
print("billing   duplicados:", billing.count()   - billing.select("invoice_id").distinct().count())

## 7. Eventos de uso (JSONL)

Los eventos vienen repartidos en 120 archivos para simular la llegada en micro-lotes.
Acá los leemos todos juntos en modo batch, solo para explorarlos. En la implementación se van a procesar con **Structured Streaming**.

In [ ]:
eventos = spark.read.json(f"{LANDING}/usage_events_stream")

eventos.printSchema()
print("Cantidad de eventos:", eventos.count())

**Observación:** Spark infiere `value` como `string` y no como número.
Pasa porque en algunos eventos llega como número (`133.0`) y en otros como texto (`"133.0"`).
Es el problema de **tipos ambiguos** que menciona la consigna: en Silver hay que convertirlo a número.

### 7.1 Evolución de esquema: versión 1 vs versión 2

La consigna dice que a mitad del período aparecen los campos `carbon_kg` y `genai_tokens`. Lo verificamos.

In [ ]:
eventos.groupBy("schema_version").agg(
    F.count("*").alias("eventos"),
    F.min("timestamp").alias("desde"),
    F.max("timestamp").alias("hasta"),
    F.count("carbon_kg").alias("con_carbon_kg"),
    F.count("genai_tokens").alias("con_genai_tokens"),
).orderBy("schema_version").show(truncate=False)

`F.count(columna)` cuenta solo los valores **no nulos**. Así vemos que la versión 1 no trae `carbon_kg` ni `genai_tokens`,
y que la versión 2 empieza el 18/07/2025.

### 7.2 ¿El `event_id` es único?

In [ ]:
total_eventos = eventos.count()
ids_distintos = eventos.select("event_id").distinct().count()

print("Eventos:", total_eventos)
print("event_id distintos:", ids_distintos)

No hay duplicados en el dato crudo, pero igual vamos a deduplicar por `event_id` en el streaming:
si el pipeline se re-ejecuta, no debe cargar dos veces el mismo evento (**idempotencia**).

### 7.3 Nulos en los eventos

In [ ]:
contar_nulos(eventos, "usage_events")

`carbon_kg` y `genai_tokens` tienen muchos nulos porque no existen en la versión 1 (y `genai_tokens` solo aplica al servicio genai).
En cambio, los nulos en `value` y `unit` son **problemas de calidad**.

### 7.4 Servicios, regiones y métricas

In [ ]:
eventos.groupBy("service").count().orderBy(F.desc("count")).show()
eventos.groupBy("region").count().orderBy(F.desc("count")).show()
eventos.groupBy("metric", "unit").count().orderBy("metric").show()

### 7.5 Costos negativos y valores extremos

La consigna avisa que hay costos negativos y *spikes* (picos muy altos).

In [ ]:
negativos = eventos.filter(F.col("cost_usd_increment") < 0).count()
print("Eventos con costo negativo:", negativos)

eventos.select("cost_usd_increment").describe().show()

In [ ]:
# Percentiles: el 99% de los eventos cuesta menos que el valor p99.
# Si el máximo está muy lejos del p99, hay valores extremos (outliers).
percentiles = eventos.approxQuantile("cost_usd_increment", [0.5, 0.99], 0.001)
print("Mediana (p50):", percentiles[0])
print("Percentil 99 :", percentiles[1])

El máximo (~317 USD) es casi 20 veces el p99 (~17 USD): confirma la presencia de outliers.
Para detectarlos sin que ellos mismos distorsionen el cálculo, en Silver usaremos un método robusto (MAD o percentiles).

### 7.6 ¿Los eventos llegan ordenados?

Leemos un solo archivo y miramos el rango de fechas que contiene.

In [ ]:
un_archivo = spark.read.json(f"{LANDING}/usage_events_stream/events_part_0000.jsonl")

un_archivo.select(F.min("timestamp"), F.max("timestamp")).show(truncate=False)
un_archivo.select("timestamp").show(5, truncate=False)

Un solo archivo trae eventos de **todo** el período (julio a agosto) y desordenados.
En streaming esto significa que van a llegar **datos tardíos** (*late data*): por eso necesitaremos un **watermark**.

## 8. Chequeos de calidad puntuales

### 8.1 Facturación: tipo de cambio por moneda

Una factura en USD debería tener `exchange_rate_to_usd = 1`. Lo revisamos.

In [ ]:
billing.groupBy("currency").agg(
    F.count("*").alias("facturas"),
    F.min("exchange_rate_to_usd").alias("fx_min"),
    F.max("exchange_rate_to_usd").alias("fx_max"),
).show()

Las facturas en USD tienen tipos de cambio entre 0,85 y 1,12: es un error de los datos.
**Decisión D-01:** en Silver se fuerza el tipo de cambio a 1 para USD y se marca la fila con un flag.

In [ ]:
print("Subtotales negativos:", billing.filter(F.col("subtotal").cast("double") < 0).count())
print("Créditos nulos      :", billing.filter(F.col("credits").isNull()).count())

### 8.2 Satisfacción: CSAT (escala 1 a 5)

In [ ]:
tickets.groupBy("csat").count().orderBy("csat").show()

Aparecen valores 0, 6 y 7, que están fuera de la escala 1–5. Se tratarán como inválidos.

### 8.3 NPS (escala −100 a 100)

In [ ]:
orgs.select(F.col("nps_score").cast("double").alias("nps_score")).describe().show()
nps.select(F.col("nps_score").cast("double").alias("nps_score")).describe().show()

### 8.4 Tickets abiertos, críticos y con SLA incumplido

In [ ]:
print("Tickets abiertos (sin resolved_at):", tickets.filter(F.col("resolved_at").isNull()).count())
print("Tickets críticos                  :", tickets.filter(F.col("severity") == "critical").count())
print("Tickets con SLA incumplido        :", tickets.filter(F.col("sla_breached") == "True").count())

### 8.5 Integridad: ¿todos los eventos tienen su organización y su recurso en los maestros?

Hacemos un **left join** de los eventos contra el maestro: si algún evento no encuentra su organización,
la columna del maestro queda en nulo.

In [ ]:
orgs_ids = orgs.select("org_id", "org_name")

eventos_sin_org = (
    eventos.select("org_id").distinct()
    .join(orgs_ids, on="org_id", how="left")
    .filter(F.col("org_name").isNull())
)
print("org_id de eventos sin maestro:", eventos_sin_org.count())

In [ ]:
recursos_ids = resources.select("resource_id", "service").withColumnRenamed("service", "service_maestro")

eventos_sin_recurso = (
    eventos.select("resource_id").distinct()
    .join(recursos_ids, on="resource_id", how="left")
    .filter(F.col("service_maestro").isNull())
)
print("resource_id de eventos sin maestro:", eventos_sin_recurso.count())

### 8.6 Rango de fechas de cada fuente

In [ ]:
eventos.select(F.min("timestamp"), F.max("timestamp")).show(truncate=False)
billing.select(F.min("month"), F.max("month")).show()
tickets.select(F.min("created_at"), F.max("created_at")).show()

La facturación empieza en **junio**, pero los eventos recién el **3 de julio**.
No se puede comparar uso contra facturación en junio: queda documentado como riesgo.

## 9. Conclusiones

| Hallazgo | Qué haremos |
|---|---|
| `value` llega como texto o número | Convertirlo a número en Silver |
| Versión 1 y 2 de los eventos | Unificarlas: en v1, `carbon_kg` y `genai_tokens` quedan nulos |
| `unit` nulo en algunos eventos | Regla de calidad → quarantine |
| Costos negativos y valores extremos | Regla de calidad + flag de anomalía |
| Eventos desordenados | Watermark + deduplicación por `event_id` |
| Tipo de cambio de USD distinto de 1 | Forzar a 1 y marcar (decisión D-01) |
| CSAT y NPS fuera de rango | Marcar como inválidos |
| Facturación desde junio, eventos desde julio | Documentado como riesgo |

El detalle completo está en `docs/02_5v_e_inventario_fuentes.md`.

In [ ]:
# Cerrar la sesión de Spark al terminar
spark.stop()